In [1]:
from keras.src.losses import mean_squared_error
from sqlalchemy.dialects.mssql.information_schema import columns
from torch.backends.opt_einsum import strategy
%load_ext autoreload
%autoreload 2

import utils
from feature_engineering import FeatureEngineering
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.types import Integer, Float, DateTime, REAL

import model
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import feature_engineering
import config  # все пути и настройки проекта лежат в config.py

ДАННЫЕ ДЛЯ ОБУЧЕНИЯ

In [2]:
np.random.seed(config.RANDOM_SEED)

users = np.random.choice(a=range(1, 140000), size=10000, replace=False)
data = pd.read_parquet(config.PARQUET_DIR,
                       filters=[('user_id', 'in', users)])

data.sort_values(by='ts', inplace=True)

X, y = data.drop(columns=['rating']), data['rating']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.75, shuffle=False)
X_valid, X_test, y_valid, y_test = train_test_split(X_valid, y_valid, train_size=0.5, shuffle=False)

ОБУЧЕНИЕ БАЗОВОЙ МОДЕЛИ

In [4]:
import model

base_model = model.Model(
    model=model.BaseModel,
    strategy='mean',
    config_value=4.0
).fit(X_train, y_train)

print(base_model.get_metrics(X_train, y_train))
print(base_model.get_metrics(X_valid, y_valid))
print(base_model.get_metrics(X_test, y_test))

{'MSE': 0.9403315599736105, 'MAE': 0.762138985362344, 'MAPE': 0.3284829795543505}
{'MSE': 1.201139382038556, 'MAE': 0.7987418685244294, 'MAPE': 0.4271952952438355}
{'MSE': 1.156006444079518, 'MAE': 0.7764144892980502, 'MAPE': 0.40494581449095735}


ТЮНИНГ

In [3]:
from sklearn.metrics import mean_squared_error as MSE
from catboost import CatBoostRegressor

tun = model.Tuner(
    n_trials=50,
    direction='minimize',
    metric=MSE,
    model=CatBoostRegressor
)
F = config.FEATS_CATBOOST
tun.tuning(x_train=X_train[F], y_train=y_train, x_valid=X_valid[F], y_valid=y_valid, save_mode=True)

[I 2026-11-14 03:53:15,657] A new study created in RDB with name: find_hp_catboost


Подбор find_hp_catboost: готово попыток 0, осталось 50


[I 2026-11-14 03:53:37,520] Trial 0 finished with value: 0.6856993297253938 and parameters: {'depth': 6, 'learning_rate': 0.2678154091306088, 'l2_leaf_reg': 0.8471801418819978}. Best is trial 0 with value: 0.6856993297253938.
[I 2026-11-14 03:55:04,120] Trial 1 finished with value: 0.6816522302493542 and parameters: {'depth': 8, 'learning_rate': 0.04296748115469074, 'l2_leaf_reg': 0.004207053950287938}. Best is trial 1 with value: 0.6816522302493542.
[I 2026-11-14 03:55:41,925] Trial 2 finished with value: 0.6861244338443706 and parameters: {'depth': 4, 'learning_rate': 0.22044355215811345, 'l2_leaf_reg': 0.2537815508265665}. Best is trial 1 with value: 0.6816522302493542.
[I 2026-11-14 03:58:51,720] Trial 3 finished with value: 0.6777589657851755 and parameters: {'depth': 8, 'learning_rate': 0.031456163175583855, 'l2_leaf_reg': 7.579479953348009}. Best is trial 3 with value: 0.6777589657851755.
[I 2026-11-14 04:00:13,001] Trial 4 finished with value: 0.6812378720302967 and parameters:

ОБУЧЕНИЕ БУСТИНГА (CATBOOST)

In [3]:
from catboost import CatBoostRegressor
import json

F = config.FEATS_CATBOOST
params = json.load(open(config.PARAMS_FILES['catboost'], encoding='utf-8'))

cat_model = model.Model(
    model=CatBoostRegressor,
    **params, verbose=False
).fit(X_train[F], y_train, save_mode=True)

print(cat_model.get_metrics(X_train[F], y_train, save_mode=True))
print(cat_model.get_metrics(X_valid[F], y_valid, save_mode=True))
print(cat_model.get_metrics(X_test[F], y_test, save_mode=True))


{'MSE': 0.6380756297944907, 'MAE': 0.6181194265581665, 'MAPE': 0.2552780446997678}
{'MSE': 0.6772102843780493, 'MAE': 0.6248233322300863, 'MAPE': 0.2875205775752045}
{'MSE': 0.685116042083168, 'MAE': 0.6208104397267533, 'MAPE': 0.2839077220654699}


In [6]:
d = pd.DataFrame(
    {'a':[1], 'b':[0]}
)


print(d.iloc[0].to_dict())

{'a': 1, 'b': 0}
